In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/funnykids
/kaggle/input/datasets/funnykids/face2face-frame32
/kaggle/input/datasets/funnykids/face2face-frame32/Face2Face_frame32
/kaggle/input/datasets/funnykids/face2face-frame32/original_frames32


In [2]:
# =========================================================
# DeepFake Detection - CNN + BiLSTM  [IMPROVED VERSION]
# TRAIN: 600 | TEST: 100
#
# IMPROVEMENTS:
#   1. CNN Feature Extractor (MobileNetV3) before LSTM
#      → LSTM now gets real semantic features, not raw pixels
#   2. Bidirectional LSTM with residual connections
#   3. Multi-Head Self-Attention on LSTM output
#   4. Focal Loss (handles hard negatives better)
#   5. Cosine Annealing + Warmup Scheduler
#   6. Early Stopping
#   7. Gradient Clipping
#   8. Stronger Data Augmentation
#   9. Test-Time Augmentation
#  10. AUC + F1 metrics
# =========================================================

# =========================================================
# INSTALLATIONS
# =========================================================
# !pip uninstall -y torch torchvision torchaudio -q
# !pip install -q torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 \
#     --index-url https://download.pytorch.org/whl/cu118
# !pip install -q timm==0.9.12 opencv-python

# =========================================================
# IMPORTS
# =========================================================
import os
import cv2
import numpy as np
import copy
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, f1_score
)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
import timm

# =========================================================
# REPRODUCIBILITY
# =========================================================
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True

# =========================================================
# DEVICE
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

# =========================================================
# DATA PATHS
# =========================================================
REAL_PATH = "/kaggle/input/datasets/funnykids/face2face-frame32/original_frames32"
FAKE_PATH = "/kaggle/input/datasets/funnykids/face2face-frame32/Face2Face_frame32"

# =========================================================
# SETTINGS
# =========================================================
IMG_SIZE      = 112         # larger than original 64 → more detail for CNN
PATCH_SIZE    = 16          # CNN processes image as sequence of patches
SEQ_LEN       = (IMG_SIZE // PATCH_SIZE) ** 2  # 49 patches

BATCH_SIZE    = 16
EPOCHS        = 30
LR            = 5e-4
WEIGHT_DECAY  = 1e-4
PATIENCE      = 7
GRAD_CLIP     = 1.0

TRAIN_IMAGES  = 600
TEST_IMAGES   = 100
TOTAL_IMAGES  = TRAIN_IMAGES + TEST_IMAGES

# =========================================================
# LOAD DATA
# =========================================================
def load_images(path, count, label):
    imgs, lbls = [], []
    files = sorted(os.listdir(path))
    for f in tqdm(files[:count], desc=f"Loading {'REAL' if label==0 else 'FAKE'}"):
        fp  = os.path.join(path, f)
        img = cv2.imread(fp)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        imgs.append(img)
        lbls.append(label)
    return imgs, lbls

real_imgs, real_lbls = load_images(REAL_PATH, TOTAL_IMAGES // 2, 0)
fake_imgs, fake_lbls = load_images(FAKE_PATH, TOTAL_IMAGES // 2, 1)

images = np.array(real_imgs + fake_imgs)
labels = np.array(real_lbls + fake_lbls)

print(f"\nTOTAL: {len(images)}  (real={sum(labels==0)}, fake={sum(labels==1)})")

# =========================================================
# SPLIT
# =========================================================
X_train, X_test, y_train, y_test = train_test_split(
    images, labels,
    test_size=TEST_IMAGES / TOTAL_IMAGES,
    random_state=SEED, stratify=labels
)
print(f"TRAIN: {len(X_train)}  |  TEST: {len(X_test)}")

# =========================================================
# TRANSFORMS
# =========================================================
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2)
])

test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

tta_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# =========================================================
# DATASET
# =========================================================
class DeepFakeDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images    = images
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img   = self.images[idx]
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.long)

# =========================================================
# WEIGHTED SAMPLER
# =========================================================
class_counts   = np.bincount(y_train)
sample_weights = (1.0 / class_counts)[y_train]
sampler = WeightedRandomSampler(
    torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

train_dataset = DeepFakeDataset(X_train, y_train, train_transform)
test_dataset  = DeepFakeDataset(X_test,  y_test,  test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          sampler=sampler, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False,  num_workers=2, pin_memory=True)

# =========================================================
# FOCAL LOSS
# =========================================================
class FocalLoss(nn.Module):
    """
    Focuses training on hard / misclassified examples.
    gamma=2 is a common default.
    """
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.1):
        super().__init__()
        self.gamma           = gamma
        self.alpha           = alpha
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(
            inputs, targets,
            reduction="none",
            label_smoothing=self.label_smoothing
        )
        pt   = torch.exp(-ce_loss)
        loss = ((1 - pt) ** self.gamma) * ce_loss
        return loss.mean()

# =========================================================
# PATCH EMBEDDING  (turns image into sequence for LSTM)
# =========================================================
class PatchEmbedding(nn.Module):
    """
    Splits image into fixed-size patches and projects each
    patch to an embedding vector using a CNN stem.
    Input : [B, 3, H, W]
    Output: [B, SEQ_LEN, EMBED_DIM]
    """
    def __init__(self, img_size=IMG_SIZE, patch_size=PATCH_SIZE, embed_dim=256):
        super().__init__()
        self.patch_size = patch_size
        n_patches       = (img_size // patch_size) ** 2

        # Small CNN to extract per-patch features
        self.proj = nn.Sequential(
            nn.Conv2d(3, 64,  kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.GELU(),
            nn.Conv2d(128, embed_dim, kernel_size=patch_size, stride=patch_size)
        )
        # Positional embedding
        self.pos_embed = nn.Parameter(torch.randn(1, n_patches, embed_dim) * 0.02)

    def forward(self, x):
        x = self.proj(x)                          # [B, D, H/P, W/P]
        B, D, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)          # [B, N, D]
        x = x + self.pos_embed                    # positional encoding
        return x

# =========================================================
# MULTI-HEAD SELF-ATTENTION (on LSTM output)
# =========================================================
class SelfAttentionPool(nn.Module):
    """Attend over LSTM timesteps and produce a single vector."""
    def __init__(self, hidden_dim, n_heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=n_heads,
            batch_first=True,
            dropout=0.1
        )
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x):                         # x: [B, T, D]
        attn_out, _ = self.attn(x, x, x)
        x = self.norm(x + attn_out)               # residual
        return x.mean(dim=1)                      # mean pool → [B, D]

# =========================================================
# IMPROVED CNN + BiLSTM MODEL
# =========================================================
class ImprovedCNNLSTM(nn.Module):
    def __init__(self):
        super().__init__()

        EMBED_DIM  = 256
        LSTM_H     = 256
        N_LAYERS   = 2
        N_HEADS    = 4

        # 1. Patch embedding: image → sequence of feature vectors
        self.patch_embed = PatchEmbedding(
            img_size   = IMG_SIZE,
            patch_size = PATCH_SIZE,
            embed_dim  = EMBED_DIM
        )

        # 2. Bidirectional LSTM
        self.lstm = nn.LSTM(
            input_size   = EMBED_DIM,
            hidden_size  = LSTM_H,
            num_layers   = N_LAYERS,
            batch_first  = True,
            dropout      = 0.3,
            bidirectional= True
        )

        lstm_out_dim = LSTM_H * 2   # bidirectional

        # 3. Self-attention pooling
        self.attn_pool = SelfAttentionPool(lstm_out_dim, n_heads=N_HEADS)

        # 4. Classifier
        self.classifier = nn.Sequential(
            nn.Linear(lstm_out_dim, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        # x: [B, 3, H, W]
        seq = self.patch_embed(x)              # [B, N, EMBED]
        lstm_out, _ = self.lstm(seq)           # [B, N, LSTM_H*2]
        pooled = self.attn_pool(lstm_out)      # [B, LSTM_H*2]
        return self.classifier(pooled)

# =========================================================
# BUILD MODEL
# =========================================================
model = ImprovedCNNLSTM().to(device)

total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total:,}")
print(model)

# =========================================================
# LOSS + OPTIMIZER + SCHEDULER
# =========================================================
criterion = FocalLoss(gamma=2.0, label_smoothing=0.1)

optimizer = optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

# Warmup for first 3 epochs, then cosine decay
def lr_lambda(epoch):
    warmup = 3
    if epoch < warmup:
        return (epoch + 1) / warmup
    progress = (epoch - warmup) / (EPOCHS - warmup)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# =========================================================
# EARLY STOPPING
# =========================================================
class EarlyStopping:
    def __init__(self, patience=7, min_delta=1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best_loss  = None
        self.best_state = None
        self.stop       = False

    def step(self, val_loss, model):
        if self.best_loss is None or val_loss < self.best_loss - self.min_delta:
            self.best_loss  = val_loss
            self.best_state = copy.deepcopy(model.state_dict())
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

early_stopping = EarlyStopping(patience=PATIENCE)

# =========================================================
# TRAINING
# =========================================================
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

print("\n" + "="*55)
print("STARTING TRAINING")
print("="*55)

for epoch in range(EPOCHS):

    # ---- TRAIN ----
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for imgs, lbls in tqdm(train_loader, desc=f"Epoch [{epoch+1}/{EPOCHS}] TRAIN"):
        imgs, lbls = imgs.to(device), lbls.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, lbls)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(lbls.cpu().numpy())

    train_loss = running_loss / len(train_loader)
    train_acc  = accuracy_score(all_labels, all_preds)

    # ---- EVAL ----
    model.eval()
    val_loss = 0.0
    val_preds, val_labels = [], []

    with torch.no_grad():
        for imgs, lbls in tqdm(test_loader, desc=f"Epoch [{epoch+1}/{EPOCHS}] EVAL "):
            imgs, lbls = imgs.to(device), lbls.to(device)
            outputs  = model(imgs)
            loss     = criterion(outputs, lbls)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(lbls.cpu().numpy())

    val_loss /= len(test_loader)
    val_acc   = accuracy_score(val_labels, val_preds)
    val_f1    = f1_score(val_labels, val_preds, average="weighted")

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    current_lr = optimizer.param_groups[0]["lr"]
    print(f"\n{'='*55}")
    print(f"EPOCH {epoch+1}/{EPOCHS}  |  LR: {current_lr:.2e}")
    print(f"  Train  →  Loss: {train_loss:.4f}  |  Acc: {train_acc:.4f}")
    print(f"  Val    →  Loss: {val_loss:.4f}  |  Acc: {val_acc:.4f}  |  F1: {val_f1:.4f}")

    scheduler.step()
    early_stopping.step(val_loss, model)

    if early_stopping.stop:
        print(f"\n⚡ Early stopping at epoch {epoch+1}")
        break

model.load_state_dict(early_stopping.best_state)
print("\n✅ Best weights restored.")

# =========================================================
# TEST-TIME AUGMENTATION
# =========================================================
TTA_RUNS = 5
print(f"\nRunning TTA ({TTA_RUNS} passes)...")

model.eval()
tta_probs = np.zeros((len(X_test), 2))

for _ in range(TTA_RUNS):
    tta_ds = DeepFakeDataset(X_test, y_test, tta_transform)
    tta_ld = DataLoader(tta_ds, batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=2)
    with torch.no_grad():
        start = 0
        for imgs, _ in tta_ld:
            imgs  = imgs.to(device)
            probs = F.softmax(model(imgs), dim=1).cpu().numpy()
            end   = start + len(imgs)
            tta_probs[start:end] += probs
            start = end

tta_probs /= TTA_RUNS
tta_preds  = np.argmax(tta_probs, axis=1)

# =========================================================
# FINAL RESULTS
# =========================================================
acc = accuracy_score(y_test, tta_preds)
f1  = f1_score(y_test, tta_preds, average="weighted")
try:
    auc = roc_auc_score(y_test, tta_probs[:, 1])
except Exception:
    auc = float("nan")

print("\n" + "="*55)
print("FINAL TEST RESULTS (with TTA)")
print("="*55)
print(f"  Accuracy  : {acc:.4f}")
print(f"  F1 Score  : {f1:.4f}")
print(f"  ROC-AUC   : {auc:.4f}")
print()
print(classification_report(y_test, tta_preds, target_names=["REAL", "FAKE"]))
print("CONFUSION MATRIX:")
print(confusion_matrix(y_test, tta_preds))

# =========================================================
# SAVE
# =========================================================
torch.save({
    "model_state_dict": model.state_dict(),
    "history": history,
    "accuracy": acc,
    "f1": f1,
    "auc": auc
}, "cnn_lstm_improved.pth")

print("\nMODEL SAVED → cnn_lstm_improved.pth")

DEVICE: cpu


Loading FAKE: 100%|██████████| 350/350 [00:01<00:00, 232.76it/s]



TOTAL: 700  (real=350, fake=350)
TRAIN: 600  |  TEST: 100

Trainable parameters: 12,307,266
ImprovedCNNLSTM(
  (patch_embed): PatchEmbedding(
    (proj): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): GELU(approximate='none')
      (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): GELU(approximate='none')
      (6): Conv2d(128, 256, kernel_size=(16, 16), stride=(16, 16))
    )
  )
  (lstm): LSTM(256, 256, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (attn_pool): SelfAttentionPool(
    (attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
    )
    (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (classifier): Seq

Epoch [1/30] TRAIN:   0%|          | 0/38 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Epoch [1/30] EVAL :   0%|          | 0/7 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Epoch [1/30] EVAL : 100%|██████████| 7/7 [00:05<00:00,  1.34it/s]



EPOCH 1/30  |  LR: 1.67e-04
  Train  →  Loss: 0.2116  |  Acc: 0.5417
  Val    →  Loss: 0.1784  |  Acc: 0.5000  |  F1: 0.4430


...

Epoch [10/30] TRAIN:   0%|          | 0/38 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Epoch [10/30] EVAL :   0%|          | 0/7 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Epoch [10/30] EVAL : 100%|██████████| 7/7 [00:04<00:00,  1.49it/s]



EPOCH 10/30  |  LR: 4.42e-04
  Train  →  Loss: 0.1830  |  Acc: 0.5050
  Val    →  Loss: 0.1728  |  Acc: 0.5100  |  F1: 0.5060


Epoch [11/30] TRAIN:   0%|          | 0/38 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Epoch [11/30] EVAL :   0%|          | 0/7 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Epoch [11/30] EVAL : 100%|██████████| 7/7 [00:05<00:00,  1.29it/s]


EPOCH 11/30  |  LR: 4.22e-04
  Train  →  Loss: 0.1843  |  Acc: 0.4917
  Val    →  Loss: 0.1772  |  Acc: 0.5100  |  F1: 0.4826

⚡ Early stopping at epoch 11

✅ Best weights restored.

Running TTA (5 passes)...



FINAL TEST RESULTS (with TTA)
  Accuracy  : 0.6000
  F1 Score  : 0.5994
  ROC-AUC   : 0.6056

              precision    recall  f1-score   support

        REAL       0.61      0.56      0.58        50
        FAKE       0.59      0.64      0.62        50

    accuracy                           0.60       100
   macro avg       0.60      0.60      0.60       100
weighted avg       0.60      0.60      0.60       100

CONFUSION MATRIX:
[[28 22]
 [18 32]]

MODEL SAVED → cnn_lstm_improved.pth
